# Spam Classifier

## Preparing the Train/Test Data

### Hyperparameters

In [57]:
strip_off_email_headers = True
lower_case = True
remove_punctuation = True
replace_urls_with_URL = True
replace_numbers_with_NUMBER = True
do_stemming = True

### Count Unique Words

In [58]:
import os
import random
import re
import string

import nltk
from nltk.stem import PorterStemmer

from email import policy
from email.parser import BytesParser
from bs4 import BeautifulSoup

folders = ["20030228_hard_ham\\hard_ham", "20030228_easy_ham\\easy_ham", "20030228_easy_ham_2\\easy_ham_2",
          "20030228_spam\\spam", "20030228_spam_2\\spam_2"]

exclude = {"cmds"}
unique_words = set()

for folder in folders:

    files = [
    f for f in os.listdir(folder)
    if f not in exclude
    ]

    random.seed(42)
    random.shuffle(files)
    
    for file in files:
        with open(folder + "\\" + file, "rb") as f:
            msg = BytesParser(policy=policy.default).parse(f)

        # Extract only the headers you actually need
        subject = msg.get("Subject", "")
        sender = msg.get("From", "")
        date = msg.get("Date", "")

        headers = "\n".join([
        f"Subject: {subject}",
        f"From: {sender}",
        f"Date: {date}"
        ])
        # Body
        body = msg.get_body(preferencelist=("html", "plain"))

        if body:
            payload = body.get_payload(decode=True)

            charset = body.get_content_charset()

            if not charset or charset.upper() in ("DEFAULT", "UNKNOWN-8BIT"):
                charset = "utf-8"

            try:
                content = payload.decode(charset, errors="replace")
            except (LookupError, UnicodeDecodeError):
                content = payload.decode("latin-1", errors="replace")

            if body.get_content_type() == "text/html":
                content = BeautifulSoup(content, "html.parser").get_text(
                separator="\n", strip=True
            )
        else:
            content = ""

        # Combine headers and body
        text = ""
        if not strip_off_email_headers:
            if remove_punctuation:
                headers = headers.translate(str.maketrans("", "", string.punctuation))
            if replace_urls_with_URL:
                headers = re.sub(
                        r'https?://\S+|www\.\S+',
                        'URL',
                        headers
                        )
            if replace_numbers_with_NUMBER:
                headers = re.sub(r'\b\d+(?:\.\d+)?\b', 'NUMBER', headers)
            text += headers + "\n"

        if remove_punctuation:
            content = content.translate(str.maketrans("", "", string.punctuation))
        if replace_urls_with_URL:
                content = re.sub(
                        r'https?://\S+|www\.\S+',
                        'URL',
                        content
                        )
        if replace_numbers_with_NUMBER:
                content = re.sub(r'\b\d+(?:\.\d+)?\b', 'NUMBER', content)
        text += content

        # Extract words and add them to the set
        if lower_case:
            words = re.findall(r"\b\w+\b", text.lower())
        else:
            words = re.findall(r"\b\w+\b", text)

        if do_stemming:
            stemmer = PorterStemmer()
            stemmed_words = [stemmer.stem(word) for word in words]
            unique_words.update(stemmed_words)
        else:
            unique_words.update(words)


### Create Feature Vectors

In [59]:
import os
import random
import re
import string

import nltk
from nltk.stem import PorterStemmer

from email import policy
from email.parser import BytesParser
from bs4 import BeautifulSoup

folders = ["20030228_hard_ham\\hard_ham", "20030228_easy_ham\\easy_ham", "20030228_easy_ham_2\\easy_ham_2",
          "20030228_spam\\spam", "20030228_spam_2\\spam_2"]

exclude = {"cmds"}

X_train = []
X_test = []
y_train = []
y_test = []

words_list = list(unique_words)

for folder in folders:

    files = [
    f for f in os.listdir(folder)
    if f not in exclude
    ]

    random.seed(42)
    random.shuffle(files)

    length_first_80 = len(files) * 80 / 100
    
    for i in range(len(files)):
        with open(folder + "\\" + files[i], "rb") as f:
            msg = BytesParser(policy=policy.default).parse(f)

        # Extract only the headers you actually need
        subject = msg.get("Subject", "")
        sender = msg.get("From", "")
        date = msg.get("Date", "")

        headers = "\n".join([
        f"Subject: {subject}",
        f"From: {sender}",
        f"Date: {date}"
        ])
        
        # Body
        body = msg.get_body(preferencelist=("html", "plain"))

        if body:
            payload = body.get_payload(decode=True)

            charset = body.get_content_charset()

            if not charset or charset.upper() in ("DEFAULT", "UNKNOWN-8BIT"):
                charset = "utf-8"

            try:
                content = payload.decode(charset, errors="replace")
            except (LookupError, UnicodeDecodeError):
                content = payload.decode("latin-1", errors="replace")

            if body.get_content_type() == "text/html":
                content = BeautifulSoup(content, "html.parser").get_text(
                separator="\n", strip=True
            )
        else:
            content = ""

        # Combine headers and body
        text = ""
        if not strip_off_email_headers:
            if remove_punctuation:
                headers = headers.translate(str.maketrans("", "", string.punctuation))
            if replace_urls_with_URL:
                headers = re.sub(
                        r'https?://\S+|www\.\S+',
                        'URL',
                        headers
                        )
            if replace_numbers_with_NUMBER:
                headers = re.sub(r'\b\d+(?:\.\d+)?\b', 'NUMBER', headers)
            text += headers + "\n"

        if remove_punctuation:
            content = content.translate(str.maketrans("", "", string.punctuation))
        if replace_urls_with_URL:
                content = re.sub(
                        r'https?://\S+|www\.\S+',
                        'URL',
                        content
                        )
        if replace_numbers_with_NUMBER:
                content = re.sub(r'\b\d+(?:\.\d+)?\b', 'NUMBER', content)
        text += content

        # Extract words and add them to the set
        if lower_case:
            words = re.findall(r"\b\w+\b", text.lower())
        else:
            words = re.findall(r"\b\w+\b", text)

        if do_stemming:
            stemmer = PorterStemmer()
            stemmed_words = [stemmer.stem(word) for word in words]
            words = stemmed_words
            
        feature_vector = [0] * len(unique_words)

        for word in words:
            index = words_list.index(word)
            feature_vector[index] = feature_vector[index] + 1
            
        if i < length_first_80:
            X_train.append(feature_vector)
            if "spam" in folder:
                y_train.append(1)
            else:
                y_train.append(0)
        
        else:
            X_test.append(feature_vector)
            if "spam" in folder:
                y_test.append(1)
            else:
                y_test.append(0)

## Standardize Feature Vectors

In [64]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

## Training

### Random Forest Classifier

In [61]:
from sklearn.ensemble import RandomForestClassifier
forest_clf = RandomForestClassifier(random_state=42)

### Measuring accuracy using cross-validation

In [65]:
from sklearn.model_selection import cross_val_score
cross_val_score(forest_clf, X_train_scaled, y_train, cv=5, scoring="accuracy").mean()

np.float64(0.956601741776133)

## Testing (Precision and Recall)

In [70]:
scaler = StandardScaler()
X_test_scaled = scaler.fit_transform(X_test)

forest_clf.fit(X_train_scaled, y_train)
predictions = forest_clf.predict(X_test_scaled)

from sklearn.metrics import precision_score, recall_score

print(precision_score(y_test, predictions))
print(recall_score(y_test, predictions))

0.9828571428571429
0.9076517150395779
